In [ ]:
import rasterio
import numpy as np
from collections import Counter


In [ ]:
cd "../data/OUT8"

In [ ]:

flags_path = "S2A_MSI_2016_01_18_04_42_31_T45QXE_BoB_L2W_l2_flags.tif"

with rasterio.open(flags_path) as src:
    flags = src.read(1)

# flatten valid pixels
flags_flat = flags[~np.isnan(flags)].astype(int)

# count unique flag values
counts = Counter(flags_flat)

print("Unique flag values and pixel counts:")
for k, v in sorted(counts.items()):
    print(f"Flag {k:>3}: {v:,} pixels")


In [ ]:
def has_flag(arr, bit):
    return (arr & (1 << bit)) != 0

mask_outofscene = has_flag(flags, 4)
mask_cirrus     = has_flag(flags, 1)
mask_swir       = has_flag(flags, 0)
mask_negative   = has_flag(flags, 3)
mask_high_toa   = has_flag(flags, 2)

in_scene = ~mask_outofscene

usable = (
    has_flag(flags, 0) &    # valid water context
    ~mask_outofscene &
    ~mask_cirrus &
    ~mask_high_toa &
    ~mask_negative
)
 

print("Out of scene pixels:", np.sum(mask_outofscene))
print("Cirrus pixels:", np.sum(mask_cirrus))
print("SWIR / glint pixels:", np.sum(mask_swir))
print("Negative reflectance pixels:", np.sum(mask_negative))
print("High TOA pixels:", np.sum(mask_high_toa))
print("Usable (flag == 0) pixels:", np.sum(usable))


In [ ]:

in_scene = ~mask_outofscene
total_in_scene = np.sum(in_scene)


usable_strict_in_scene = usable & in_scene
n_usable_strict = np.sum(usable_strict_in_scene)

pct_strict = 100 * n_usable_strict / total_in_scene


print(f"Total in-scene pixels: {total_in_scene:,}")

print(f"Usable: {n_usable_strict:,} "
      f"({pct_strict:.2f} %)")


